# Phase 5 — FLUX.2 Klein reference-consistency test

Creates one mascot master, then conditions 19 separate candidate images on that exact master: 3 views, 6 expressions, 5 poses and 5 scene backgrounds. This is an experiment, not an automatic identity approval. All outputs remain unapproved.

The notebook refuses to run the edit batch unless the installed `Flux2KleinPipeline.__call__` signature explicitly supports `image`. It does not use an undocumented `strength` argument.


In [ ]:
import os, sys, json, time, subprocess, pathlib, hashlib, traceback, uuid, resource, gc, inspect
from PIL import Image, ImageOps, ImageDraw
RUN_ID = globals().get('RUN_ID', 'manual-' + uuid.uuid4().hex)
RUN_TOKEN = globals().get('RUN_TOKEN', 'manual')
COMMIT_SHA = globals().get('COMMIT_SHA', 'unknown')
WORKFLOW_RUN_ID = globals().get('WORKFLOW_RUN_ID', 'local')
OUT = pathlib.Path('/kaggle/working/outputs'); OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME','/tmp/hf_home')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
MODEL_ID='black-forest-labs/FLUX.2-klein-4B'
MODEL_REVISION='e7b7dc27f91deacad38e78976d1f2b499d76a294'
WIDTH, HEIGHT, STEPS, GUIDANCE = 576, 1024, 4, 1.0
PIP_PACKAGES=['diffusers==0.42.1','transformers==5.22.1','huggingface-hub==1.33.0','accelerate','psutil','pillow']
BASE='A single simple Stickman Thoughts mascot: a large circular white head about one third of total height, thick dark charcoal outline, two small black oval eyes, expressive black eyebrows, a small simple mouth, thin black stick arms and thin black stick legs. Absolutely no torso, no clothes, no gloves, no shoes, no fingers. Flat 2D vector cartoon, soft muted colours, simple flat shading, clean readable silhouette.'
MASTER_PROMPT=f'Create a clean character reference master on a plain warm cream background. {BASE} Front view, neutral expression, standing upright, full character visible, centered, generous empty margin. No text, no watermark, no props, no extra characters.'
EDIT_CASES=[]
for view in ['front_view','side_view','three_quarter_view']:
    angle={'front_view':'front-facing','side_view':'clear side profile facing right','three_quarter_view':'three-quarter view facing slightly right'}[view]
    EDIT_CASES.append({'name':view,'seed':31001+len(EDIT_CASES),'prompt':f'Keep the exact same mascot identity, head circle, eye spacing, outline weight and thin stick limbs from the reference image. Change only the viewing angle to {angle}. Full character visible on plain warm cream background. {BASE} No redesign.'})
for expr in ['neutral','happy','sad','surprised','thinking','scared']:
    EDIT_CASES.append({'name':'expression_'+expr,'seed':31100+len(EDIT_CASES),'prompt':f'Keep the exact same mascot from the reference image. Change only the facial expression to {expr}; preserve the circular head, eye positions, outline and stick limbs. Plain warm cream background. {BASE} No redesign.'})
for pose in ['standing','walking','running','pointing','sitting']:
    EDIT_CASES.append({'name':'pose_'+pose,'seed':31200+len(EDIT_CASES),'prompt':f'Keep the exact same mascot identity from the reference image. Change only the pose to {pose}; keep the same circular head, face, outline and thin stick-limb construction. Plain warm cream background. {BASE} No redesign.'})
SCENES=[('scene_space','Standing on the moon beside a small crater, Earth in the distance'),('scene_ocean','Standing on the deck of an old ocean liner at night, dark ocean and stars'),('scene_forest','Standing in a quiet green forest with tall trees'),('scene_lab','Standing in a bright simple science laboratory with a few readable instruments'),('scene_city','Standing on a city rooftop at sunset with a distant skyline')]
for name,scene in SCENES:
    EDIT_CASES.append({'name':name,'seed':31300+len(EDIT_CASES),'prompt':f'Change the background to this scene: {scene}. Preserve the exact same mascot from the reference image, including head shape, face, outline thickness and stick limbs. The mascot must remain clearly visible and recognisable, full character in frame. {BASE} Do not turn the mascot into a human or add clothing.'})
REPORT={'run_token':RUN_TOKEN,'run_id':RUN_ID,'commit_sha':COMMIT_SHA,'workflow_run_id':WORKFLOW_RUN_ID,'status':'running','error_message':None,'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'install_ok':False,'model_downloaded':False,'pipeline_loaded':False,'cuda_available':False,'reference_conditioning_supported':False,'reference_image_sha256':None,'results':[],'planned_edit_count':len(EDIT_CASES)}
STARTED=time.monotonic()


In [ ]:
import urllib.request
REFERENCE_ASSET_MANIFEST = {"stickman_front":{"path":"cartoon/character_refs/stickman_front.png","blob_sha":"00067559e2f5cd15a349c122bd8b44881685ae1b","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/stickman_front.png"},"stickman_side":{"path":"cartoon/character_refs/stickman_side.png","blob_sha":"be7370cb06d1ddb5ca9870fb2ef2d42c7f7f6936","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/stickman_side.png"},"stickman_three_quarter":{"path":"cartoon/character_refs/stickman_three_quarter.png","blob_sha":"df2b8ebe668af0ab14924fd84bc8fac3f1a86ade","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/stickman_three_quarter.png"},"stickman_thinking":{"path":"cartoon/character_refs/stickman_thinking.png","blob_sha":"2462679150c92781a500ce137409b54194949206","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/stickman_thinking.png"},"stickman_faces":{"path":"cartoon/character_refs/stickman_faces.png","blob_sha":"587df961bf22b9d4343b5c21c8a6cde6a7352d75","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/stickman_faces.png"},"stickman_poses":{"path":"cartoon/character_refs/stickman_poses.png","blob_sha":"d228c74e3f31a38473dac3f1781bea262fb23351","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/stickman_poses.png"},"costume_ancient_egypt":{"path":"cartoon/character_refs/costumes/costume_ancient_egypt.png","blob_sha":"19e97c1432e14058185e99790f93bfeb822064e9","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/costumes/costume_ancient_egypt.png"},"costume_astronaut":{"path":"cartoon/character_refs/costumes/costume_astronaut.png","blob_sha":"18d795bd190db3746c49d84f14340f045960f24b","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/costumes/costume_astronaut.png"},"costume_diver":{"path":"cartoon/character_refs/costumes/costume_diver.png","blob_sha":"95b51baa9192cb9068c10d61c41aa419d0e851a7","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/costumes/costume_diver.png"},"costume_doctor":{"path":"cartoon/character_refs/costumes/costume_doctor.png","blob_sha":"16d49f6d0a74eaa9e1abcd768310dc1b47c106b3","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/costumes/costume_doctor.png"},"costume_explorer":{"path":"cartoon/character_refs/costumes/costume_explorer.png","blob_sha":"b48c24f25f507004096c2a835671d24e7c6be9cf","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/costumes/costume_explorer.png"},"costume_sailor":{"path":"cartoon/character_refs/costumes/costume_sailor.png","blob_sha":"09610a75d85e703dae12a17a20d1b5add8a6c737","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/costumes/costume_sailor.png"},"costume_soldier":{"path":"cartoon/character_refs/costumes/costume_soldier.png","blob_sha":"b3fc9cc3a66dae5c67525637b6fa21feeae461e3","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/costumes/costume_soldier.png"},"costume_winter":{"path":"cartoon/character_refs/costumes/costume_winter.png","blob_sha":"2820e88d549dada669d96483bc10ff7e90d0a925","url":"https://raw.githubusercontent.com/damarlanandu2005-jpg/yt-auto/f7da39dbb7506997e4da4a0753938a82e396ba17/cartoon/character_refs/costumes/costume_winter.png"}}
REFERENCE_DIR = pathlib.Path('/kaggle/working/character_references'); REFERENCE_DIR.mkdir(parents=True, exist_ok=True)
def download_character_references():
    downloaded = {}
    for name, meta in REFERENCE_ASSET_MANIFEST.items():
        target = REFERENCE_DIR / (name + '.png')
        if not target.exists():
            urllib.request.urlretrieve(meta['url'], target)
        with Image.open(target) as im:
            im.verify()
        downloaded[name] = target
    return downloaded
def reference_for_case(case_name, assets, master_path):
    if case_name == 'side_view': key = 'stickman_side'
    elif case_name == 'three_quarter_view': key = 'stickman_three_quarter'
    elif case_name.startswith('expression_'): key = 'stickman_thinking' if case_name == 'expression_thinking' else 'stickman_faces'
    elif case_name.startswith('pose_'): key = 'stickman_poses'
    else: key = 'stickman_front'
    return assets[key]

In [ ]:
def sh(cmd): return subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()
def install():
    p=sh([sys.executable,'-m','pip','install','--no-cache-dir']+PIP_PACKAGES)
    (OUT/'install.log').write_text(p.stdout,encoding='utf-8')
    if p.returncode: raise RuntimeError('pip install failed: '+p.stdout[-2500:])
    import torch, diffusers, transformers
    REPORT['torch_version']=torch.__version__; REPORT['diffusers_version']=diffusers.__version__; REPORT['transformers_version']=transformers.__version__
    REPORT['cuda_available']=bool(torch.cuda.is_available())
    REPORT['gpu_names']=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []
    REPORT['gpu_total_vram_bytes']=[int(torch.cuda.get_device_properties(i).total_memory) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []
    if not REPORT['cuda_available']: raise RuntimeError('CUDA unavailable; enable Kaggle GPU accelerator.')
    REPORT['install_ok']=True
def download():
    from huggingface_hub import snapshot_download
    t=time.monotonic(); p=snapshot_download(MODEL_ID,revision=MODEL_REVISION)
    REPORT['download_seconds']=round(time.monotonic()-t,1); REPORT['model_downloaded']=True
    (OUT/'model_metadata.json').write_text(json.dumps({'model_id':MODEL_ID,'revision':MODEL_REVISION,'source':'https://huggingface.co/'+MODEL_ID,'license':'Apache-2.0 per model card','download_seconds':REPORT['download_seconds']},indent=2))
    return p
def load(path):
    import torch
    from diffusers import Flux2KleinPipeline
    sig=inspect.signature(Flux2KleinPipeline.__call__)
    REPORT['pipeline_call_parameters']=list(sig.parameters.keys())
    if 'image' not in sig.parameters:
        raise RuntimeError('Installed Flux2KleinPipeline.__call__ has no image parameter; refusing to pretend text-to-image is reference conditioning.')
    REPORT['reference_conditioning_supported']=True
    t=time.monotonic(); pipe=Flux2KleinPipeline.from_pretrained(path,torch_dtype=torch.float16)
    pipe.enable_model_cpu_offload(gpu_id=0)
    REPORT['pipeline_loaded']=True; REPORT['pipeline_load_seconds']=round(time.monotonic()-t,1)
    return pipe
def save_result(img,name,seed,kind,source_sha=None,seconds=None):
    path=OUT/f'klein4b_{name}.png'; img.save(path)
    gray=img.convert('L'); import numpy as np
    std=float(np.asarray(gray,dtype='float32').std())
    item={'case':name,'kind':kind,'seed':seed,'file':path.name,'seconds':seconds,'image_width':img.width,'image_height':img.height,'image_sha256':sha256(path),'source_reference_sha256':source_sha,'looks_broken':std<8}
    REPORT['results'].append(item); return item
def make_contact_sheet():
    files=sorted(OUT.glob('klein4b_*.png'))
    thumbs=[]
    for p in files:
        im=Image.open(p).convert('RGB'); im.thumbnail((144,256))
        tile=Image.new('RGB',(160,286),'white'); tile.paste(im,((160-im.width)//2,5))
        ImageDraw.Draw(tile).text((5,265),p.stem.replace('klein4b_','')[:22],fill='black'); thumbs.append(tile)
    cols=5; rows=(len(thumbs)+cols-1)//cols
    sheet=Image.new('RGB',(cols*160,rows*286),(235,232,224))
    for i,tile in enumerate(thumbs): sheet.paste(tile,((i%cols)*160,(i//cols)*286))
    sheet.save(OUT/'klein4b_contact_sheet.png')


In [ ]:
try:
    install()
    model_path=download()
    pipe=load(model_path)
    reference_assets=download_character_references()
    import torch
    # Create a fresh master image, then freeze its exact bytes as the sole reference for every edit.
    t=time.monotonic(); gen=torch.Generator(device='cpu').manual_seed(30001)
    master=pipe(prompt=MASTER_PROMPT,width=WIDTH,height=HEIGHT,num_inference_steps=STEPS,guidance_scale=GUIDANCE,generator=gen).images[0]
    master_path=OUT/'klein4b_master.png'; master.save(master_path)
    REPORT['master_image_sha256']=sha256(master_path)
    REPORT['results'].append({'case':'master','kind':'text_to_image_master','seed':30001,'file':master_path.name,'seconds':round(time.monotonic()-t,2),'image_width':master.width,'image_height':master.height,'image_sha256':REPORT['master_image_sha256'],'source_reference_sha256':None,'looks_broken':False})
    # Keep the generated master for the report; each edit below selects the matching owner-approved reference sheet.
    reference=Image.open(master_path).convert('RGB'); ref_hash=sha256(master_path); REPORT['reference_image_sha256']=ref_hash
    REPORT['character_reference_source']='yt-auto cartoon/character_refs at '+ 'f7da39dbb7506997e4da4a0753938a82e396ba17'
    REPORT['downloaded_character_reference_assets']=sorted(reference_assets.keys())
    for case in EDIT_CASES:
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t=time.monotonic()
        gen=torch.Generator(device='cpu').manual_seed(case['seed'])
        case_ref_path=reference_for_case(case['name'],reference_assets,master_path)
        case_reference=Image.open(case_ref_path).convert('RGB')
        case_ref_hash=sha256(case_ref_path)
        edited=pipe(prompt=case['prompt'],image=case_reference,width=WIDTH,height=HEIGHT,num_inference_steps=STEPS,guidance_scale=GUIDANCE,generator=gen).images[0]
        torch.cuda.synchronize(); elapsed=round(time.monotonic()-t,2)
        save_result(edited,case['name'],case['seed'],'reference_edit',case_ref_hash,elapsed)
        del edited; gc.collect(); torch.cuda.empty_cache()
    REPORT['all_pictures_ok']=all(not x['looks_broken'] for x in REPORT['results'])
    REPORT['status']='success' if REPORT['all_pictures_ok'] and len(REPORT['results'])==20 else 'failed'
    if REPORT['status']!='success': REPORT['error_message']='One or more outputs look broken or the expected 20 images were not produced.'
except Exception as exc:
    REPORT['status']='failed'; REPORT['error_message']=f'{type(exc).__name__}: {exc}'; traceback.print_exc()
finally:
    REPORT['notebook_duration_seconds']=round(time.monotonic()-STARTED,2)
    try: make_contact_sheet()
    except Exception: traceback.print_exc()
    (OUT/'benchmark_report.json').write_text(json.dumps(REPORT,indent=2),encoding='utf-8')
    print(json.dumps(REPORT,indent=2))


In [ ]:
from IPython.display import display, Image as IPyImage
contact=OUT/'klein4b_contact_sheet.png'
if contact.exists(): display(IPyImage(filename=str(contact)))
print('Phase 5 report status:', REPORT.get('status'))
print('Report:', OUT/'benchmark_report.json')
print('Human review is required before approving any mascot assets.')
